# Pandas vs PySpark: MovieLens Data Processing

This notebook compares the same basic data-processing ideas in Pandas and PySpark using the two production inputs: `ratings.csv` and `movies.csv`. It is designed for presentation and learning, not as production application code.

- **Pandas** is concise and excellent for local, in-memory exploration.
- **PySpark** is designed for larger datasets and work that may be distributed across cores or machines.
- The goal is to compare APIs and execution models—not to claim that one tool is always faster.

## 1. Prepare the dataset

The project keeps large CSV files out of Git. This setup cell finds the repository root and calls the existing automatic downloader. If both files already exist, no download occurs.

In [1]:
from pathlib import Path
import sys


def find_project_root(start_path):
    """Find the repository whether Jupyter starts in root or notebooks/."""
    start_path = Path(start_path).resolve()
    for candidate in (start_path, *start_path.parents):
        if (candidate / "src" / "download_data.py").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project repository.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.download_data import ensure_movielens_data

DATA_DIR = ensure_movielens_data()
RATINGS_PATH = DATA_DIR / "ratings.csv"
MOVIES_PATH = DATA_DIR / "movies.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Ratings: {RATINGS_PATH}")
print(f"Movies:  {MOVIES_PATH}")

MovieLens dataset already exists.
Project root: /home/seans/becode/projects/movie-recommendation-pyspark
Ratings: /home/seans/becode/projects/movie-recommendation-pyspark/data/raw/ml-latest/ratings.csv
Movies:  /home/seans/becode/projects/movie-recommendation-pyspark/data/raw/ml-latest/movies.csv


## 2. Load data with Pandas

Pandas materializes a CSV in the Python process's memory. The full local ratings file can be close to 1 GB before parsing and significantly larger after Pandas creates typed columns. To keep this notebook safe on laptops and hosted notebook environments:

- files up to 250 MiB are loaded in full;
- larger files use the first 1,000,000 rating rows as a bounded teaching sample; and
- `movies.csv` is always loaded completely because it is small.

> When sampling is active, Pandas metrics describe the sample—not the full dataset. The sample is not statistically random, so it should be used to compare syntax and workflow rather than to make population claims.

In [2]:
import pandas as pd

PANDAS_FULL_LOAD_LIMIT = 250 * 1024**2  # 250 MiB
PANDAS_SAMPLE_ROWS = 1_000_000
ratings_file_size = RATINGS_PATH.stat().st_size

if ratings_file_size <= PANDAS_FULL_LOAD_LIMIT:
    ratings_pd = pd.read_csv(RATINGS_PATH)
    pandas_scope = "full ratings.csv"
else:
    ratings_pd = pd.read_csv(RATINGS_PATH, nrows=PANDAS_SAMPLE_ROWS)
    pandas_scope = f"first {len(ratings_pd):,} rows (memory-safe sample)"

movies_pd = pd.read_csv(MOVIES_PATH)

print(f"Pandas ratings scope: {pandas_scope}")
print(f"ratings.csv on disk: {ratings_file_size / 1024**2:,.1f} MiB")

Pandas ratings scope: first 1,000,000 rows (memory-safe sample)
ratings.csv on disk: 890.6 MiB


In [3]:
def pandas_overview(name, dataframe):
    memory_mib = dataframe.memory_usage(deep=True).sum() / 1024**2
    print(f"{name} shape: {dataframe.shape}")
    print(f"{name} columns: {list(dataframe.columns)}")
    print(f"{name} in-memory size: {memory_mib:,.1f} MiB")
    display(dataframe.head())


pandas_overview("ratings", ratings_pd)
pandas_overview("movies", movies_pd)

ratings shape: (1000000, 4)
ratings columns: ['userId', 'movieId', 'rating', 'timestamp']
ratings in-memory size: 30.5 MiB


,userId,movieId,rating,timestamp
0,1,1,4.0,1225734739
1,1,110,4.0,1225865086
2,1,158,4.0,1225733503
3,1,260,4.5,1225735204
4,1,356,5.0,1225735119


movies shape: (86537, 3)
movies columns: ['movieId', 'title', 'genres']
movies in-memory size: 5.2 MiB


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


## 3. Basic Pandas aggregations

These expressions execute immediately and return Python/NumPy scalar values. The displayed scope makes it clear whether the metrics use the full ratings file or the bounded sample.

In [4]:
pandas_metrics = pd.Series(
    {
        "scope": pandas_scope,
        "rating_rows": len(ratings_pd),
        "unique_users": ratings_pd["userId"].nunique(),
        "unique_movies_rated": ratings_pd["movieId"].nunique(),
        "average_rating": ratings_pd["rating"].mean(),
        "minimum_rating": ratings_pd["rating"].min(),
        "maximum_rating": ratings_pd["rating"].max(),
    },
    name="Pandas result",
)

display(pandas_metrics.to_frame())

,Pandas result
scope,"first 1,000,000 rows (memory-safe sample)"
rating_rows,1000000
unique_users,9561
unique_movies_rated,25825
average_rating,3.522055
minimum_rating,0.5
maximum_rating,5.0


## 4. Pandas `groupby` example

Count ratings per `movieId`, join the counts to movie metadata, then sort from most to least rated. If Pandas is using its bounded sample, this is the sample's top 10.

In [5]:
pandas_top_10 = (
    ratings_pd.groupby("movieId")
    .size()
    .rename("rating_count")
    .reset_index()
    .merge(movies_pd[["movieId", "title", "genres"]], on="movieId", how="inner")
    .sort_values(["rating_count", "title"], ascending=[False, True])
    .head(10)
)

display(pandas_top_10)

,movieId,rating_count,title,genres
308,318,3550,"Shawshank Redemption, The (1994)",Crime|Drama
345,356,3326,Forrest Gump (1994),Comedy|Drama|Romance|War
286,296,3164,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
2370,2571,3157,"Matrix, The (1999)",Action|Sci-Fi|Thriller
575,593,3011,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller
251,260,2858,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi
2743,2959,2497,Fight Club (1999),Action|Crime|Drama|Thriller
512,527,2492,Schindler's List (1993),Drama|War
465,480,2411,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller
4661,4993,2381,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy


## 5. Load the same CSV files with PySpark

Spark reads both production paths directly. DataFrame creation is lazy: Spark records how to read and transform the data, but most work waits until an action such as `count()`, `show()`, or `collect()` is called.

The session uses two local worker threads for a laptop-friendly demonstration. The same DataFrame API can also target a distributed Spark cluster.

In [6]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("PandasVsPySparkMovieLens")
    .master("local[2]")
    .config("spark.driver.memory", "1g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

ratings_spark = spark.read.csv(
    str(RATINGS_PATH), header=True, inferSchema=True
)
movies_spark = spark.read.csv(
    str(MOVIES_PATH), header=True, inferSchema=True
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/01 15:47:37 WARN Utils: Your hostname, DESKTOP-6I983CD, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/01 15:47:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/01 15:47:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/01 15:47:39 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [7]:
print("Ratings schema")
ratings_spark.printSchema()
print("Movies schema")
movies_spark.printSchema()

spark_rating_rows = ratings_spark.count()
spark_movie_rows = movies_spark.count()
print(f"Rating rows: {spark_rating_rows:,}")
print(f"Movie rows:  {spark_movie_rows:,}")

ratings_spark.show(5, truncate=False)
movies_spark.show(5, truncate=False)

Ratings schema
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)

Movies schema
root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)



Rating rows: 33,832,162
Movie rows:  86,537
+------+-------+------+----------+
|userId|movieId|rating|timestamp |
+------+-------+------+----------+
|1     |1      |4.0   |1225734739|
|1     |110    |4.0   |1225865086|
|1     |158    |4.0   |1225733503|
|1     |260    |4.5   |1225735204|
|1     |356    |5.0   |1225735119|
+------+-------+------+----------+
only showing top 5 rows
+-------+----------------------------------+-------------------------------------------+
|movieId|title                             |genres                                     |
+-------+----------------------------------+-------------------------------------------+
|1      |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)                    |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)           |Comedy|Romance                             |
|4      |Waiting to Exhale (1995)          |Comedy|Drama|Romance                   

## 6. Basic PySpark aggregations

One `agg` expression asks Spark for the same metrics as the Pandas section. Calling `first()` is the action that executes the plan and brings one small result row back to Python. Unlike a sampled Pandas section, these values describe the full ratings CSV.

In [8]:
spark_metrics_row = (
    ratings_spark.agg(
        F.count("*").alias("rating_rows"),
        F.countDistinct("userId").alias("unique_users"),
        F.countDistinct("movieId").alias("unique_movies_rated"),
        F.avg("rating").alias("average_rating"),
        F.min("rating").alias("minimum_rating"),
        F.max("rating").alias("maximum_rating"),
    )
    .first()
)

spark_metrics = pd.Series(
    {"scope": "full ratings.csv", **spark_metrics_row.asDict()},
    name="PySpark result",
)
display(spark_metrics.to_frame())

,PySpark result
scope,full ratings.csv
rating_rows,33832162
unique_users,330975
unique_movies_rated,83239
average_rating,3.54254
minimum_rating,0.5
maximum_rating,5.0


## 7. PySpark `groupBy` example

This is the Spark equivalent of the Pandas chain: group, count, join, sort, and limit. `show()` triggers execution and retrieves only 10 rows for display.

In [9]:
spark_top_10 = (
    ratings_spark.groupBy("movieId")
    .agg(F.count("*").alias("rating_count"))
    .join(movies_spark.select("movieId", "title", "genres"), on="movieId")
    .orderBy(F.desc("rating_count"), "title")
    .limit(10)
)

spark_top_10.show(10, truncate=False)

+-------+------------+-----------------------------------------------------+--------------------------------+
|movieId|rating_count|title                                                |genres                          |
+-------+------------+-----------------------------------------------------+--------------------------------+
|318    |122296      |Shawshank Redemption, The (1994)                     |Crime|Drama                     |
|356    |113581      |Forrest Gump (1994)                                  |Comedy|Drama|Romance|War        |
|296    |108756      |Pulp Fiction (1994)                                  |Comedy|Crime|Drama|Thriller     |
|2571   |107056      |Matrix, The (1999)                                   |Action|Sci-Fi|Thriller          |
|593    |101802      |Silence of the Lambs, The (1991)                     |Crime|Horror|Thriller           |
|260    |97202       |Star Wars: Episode IV - A New Hope (1977)            |Action|Adventure|Sci-Fi         |
|2959   |8

## 8. Syntax comparison

| Task | Pandas | PySpark |
|---|---|---|
| Load CSV | `pd.read_csv()` | `spark.read.csv()` |
| Count rows | `len(df)` | `df.count()` |
| Unique count | `df["col"].nunique()` | `df.select("col").distinct().count()` |
| Group by | `groupby()` | `groupBy()` |
| Join | `merge()` | `join()` |

Pandas syntax is often compact for interactive work. PySpark syntax makes distributed transformations explicit while preserving familiar DataFrame concepts.

## 9. Conceptual difference

### Pandas: eager, local, and in memory

Most Pandas operations execute as soon as the Python statement runs. A DataFrame normally lives in one Python process on one machine, so available RAM is a practical limit. This makes Pandas excellent for fast exploration, visualization, and moderate-sized datasets.

### Spark: lazy plans and scalable execution

Spark transformations such as `select`, `filter`, `groupBy`, and `join` build a logical plan. Spark waits for an **action**—for example `count`, `show`, `first`, or `collect`—then optimizes the plan and schedules the work. In local mode it can use multiple cores; on a cluster it can divide partitions across multiple worker machines.

> Avoid calling `collect()` on a large Spark DataFrame. It moves every result row into the driver's memory and removes Spark's scalability advantage.

## 10. Simple timing comparison

Timing is intentionally modest and transparent. The Pandas timing uses the loaded Pandas scope, which may be sampled. The Spark timing uses the full Spark DataFrame and includes job scheduling. These are **not equivalent benchmark conditions**, so compare the experience—not just the numbers.

> On this local machine, Pandas may be faster for some small operations because Spark has startup overhead. The point of Spark is scalability, not always speed on small data.

In [10]:
from time import perf_counter

pandas_start = perf_counter()
_ = ratings_pd.groupby("movieId")["rating"].mean()
pandas_seconds = perf_counter() - pandas_start

spark_start = perf_counter()
_ = ratings_spark.groupBy("movieId").agg(F.avg("rating")).count()
spark_seconds = perf_counter() - spark_start

timing = pd.DataFrame(
    {
        "tool": ["Pandas", "PySpark"],
        "data scope": [pandas_scope, "full ratings.csv"],
        "operation": ["group + mean", "group + mean + count action"],
        "seconds": [pandas_seconds, spark_seconds],
    }
)
display(timing)

,tool,data scope,operation,seconds
0,Pandas,"first 1,000,000 rows (memory-safe sample)",group + mean,0.148993
1,PySpark,full ratings.csv,group + mean + count action,20.539128


## 11. Final takeaway

- **Pandas is great for exploration and smaller datasets.** Its API is compact and its immediate execution model is intuitive.
- **PySpark is useful when data grows, processing must scale, or the same logic may run in a distributed environment.** Lazy evaluation lets Spark optimize a complete plan before execution.
- **Tool choice depends on constraints.** Dataset size, memory, deployment environment, team familiarity, and future scale matter more than a simplistic speed comparison.
- **In this project, PySpark demonstrates scalable processing over millions of MovieLens ratings.** Pandas is reserved for small final results and presentation-friendly local exploration.

In [11]:
# Release the local Spark resources when the presentation is finished.
spark.stop()